In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Frozen original step7 pixels: encoder replay, raw420 and MP4

Click **Run all**. Reuses original step7 POINT4/rgb8.pt and its saved positive raw444 roundtrip.rgb from run20261001T170749052584Z. The pixel files and raster hashes are fixed. No terminal decode or new watermark update. The later MP4 run's newly decoded pixels are not used.

Three observations use one fixed native FP32 VAE encoder:

- SAVED_RAW444: directly re-encode the exact saved positive raw444 RGB. This isolates encoder/readout replay from decoder and color-conversion replay.
- RAW420: original saved RGB8→materialized raw YUV420→reopened RGB24→VAE encode.
- MP4: the same original saved RGB8→existing libx264 MP4,CRF18,8fps,yuv420p→independent RGB24 readback→VAE encode. MP4 remains the fixed primary endpoint.

Three observations, six path and payload reads, twelve message evaluations,1044 valid costs. Three VAE encodes, zero VAE decodes, zero gradient or Transformer calls. Same g0/R44 reader, all174 valid candidate paths, both keys and both messages. No candidate deletion, threshold change, best-point selection or scan. Each branch preserves failures and the other branches can finish independently.

The original anchor had rank1 and Delta=5.4846719567056e-7. The last fresh-decode run changed its pixel raster and yielded raw444 rank2 and MP4 rank46. This run removes decoder replay variation by holding the original pixels byte-exact. Same input bytes do not force identical VAE outputs: normalized max/RMS errors and correct-key path-cost changes are reported, along with actual package versions. No hard requirement to recreate an older CUDA build or use a particular GPU.

If the anchor changes, encoder execution is implicated without decoder ambiguity. RAW420 measures the color420 transport contrast; MP4 versus RAW420 assesses additional standard codec transport. Internal color conversions are not assumed algebraically identical: commands, verbose raw logs and MP4 stream metadata are retained. This is a diagnostic contrast, not unconditional attribution to a single codec operation.

All181 frames remain320x512. Historical negative controls remain references, not fresh independent negatives. Quality is relative to marked original RGB8, saved raw444 and current raw420, not OFF. Exact repeated payload is a separate readout from path recovery. These fixed-source, fixed-phase results do not establish unknown-phase/fragment synchronization, calibrated FPR or generation-time trajectory watermarking.

Existing current-Python setup and fresh-child execution are reused. Local CPU/static validation uses simulated model execution; real pretrained VAE execution is left to the user. Source is immutable when bound below.


In [ ]:
SOURCE_SHA = 'a9d108d1012c8923f74f75c6cbf9ff8d103c59ed'
from pathlib import Path
import datetime,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Step7-Fixed-Pixels-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
REPO=Path('/content/SC-SSTW-STEP7-FIXED-PIXELS-'+STAMP)
RUN_OUTPUT=OUTPUT/'fixed_reference'
PYTHON=sys.executable
INPUT_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-Channel-Margin-Continue-V1/20261001T170749052584Z/fixed_reference')
REFERENCE_ROOT=Path('/content/drive/MyDrive/Video-WM/Zero-Mean-C1-YUV444-No-H264-V1/20261001T110124499275Z/fixed_reference')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator={'observations': 3, 'mp4_saved': 1, 'mp4_read': 1, 'path_reads': 6, 'payload_reads': 6, 'message_evaluations': 12, 'valid_costs': 1044, 'catalog_slots': 23490, 'structural_exclusions': 22446, 'reference_normalized': 3, 'reference_raw': 6, 'parent_normalized': 1, 'parent_raw': 2, 'raw420_files': 1, 'raw420_rgb_readbacks': 1},observations={p:{'status':'NOT_RUN_SETUP'} for p in ('SAVED_RAW444','RAW420','MP4')})
def write_json(path,value):
    temp=path.with_suffix(path.suffix+'.tmp');temp.write_text(json.dumps(value,indent=2)+'\n');os.replace(temp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Full writer terminal:',INPUT_ROOT,'Saved references:',REFERENCE_ROOT,'Output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind verified published source SHA before Run all')
    failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    torch_ok=logged([PYTHON,'-c','import torch,torchvision; print(torch.__version__,torchvision.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:
        logged([PYTHON,'-m','pip','install','--upgrade','torch','torchvision'],'TORCH_REPAIR')
    logged([PYTHON,'-m','pip','install','-r',str(REPO/'experiments/wan_state_clock/requirements-grow-video-reference.txt')],'DEPENDENCIES')
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual imports determine availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    import_code='import json,torch; from diffusers import AutoencoderKLWan; from experiments.wan_state_clock.zero_mean_c1_yuv444_no_h264_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',import_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    logged(['ffmpeg','-version'],'FFMPEG_PROBE');logged(['ffprobe','-version'],'FFPROBE_PROBE')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_fresh_children')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.zero_mean_step7_fixed_pixels_run','--output',str(RUN_OUTPUT),'--input-root',str(INPUT_ROOT),'--reference-root',str(REFERENCE_ROOT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; full setup denominator is retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
print('Status:',result['status'],'Counts:',result.get('counts'),'Calls:',result['calls'])
print('Saved raster matches parent:',result.get('saved_raster_matches_parent'))
print('Environment package changes:',result.get('environment_package_changes'))
print('SAVED_RAW444 encoder replay:',result.get('baseline_replay'))
for name,row in result['comparisons'].items():print(name,row)
print('Fixed endpoint:',result.get('fixed_endpoint'))
print('MP4:',result['mp4'])
for name,row in result['posthoc'].items():print(name,row)
for name,row in result['message_evaluations'].items():print(name,row)
for name,row in result['observations'].items():
    print(name,'status=',row['status'],'quality_vs_shared_q8=',row.get('quality_vs_shared_q8'),
          'quality_vs_parent_raw444=',row.get('quality_vs_parent_raw444'),
          'quality_vs_current_raw420=',row.get('quality_vs_current_raw420'))
print('Failures:',result['failures'])
print(result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
